# Agricultural Burning Control Notices
*Myriah Hodgson, University of Oregon EWP, 4/4/2025*

Documentation of compiling agricultural burning control notices, years 2020-2024.

In [163]:
# imports
import pandas as pd 
import numpy as np

Functions for cleaning individual datasheets:

In [164]:
def clean_month(df):
    """Return the cleaned dataframe by slicing and getting appropriate headings"""
    df.columns = df.iloc[3, :] # headings we want are in this row
    df = df.iloc[4:36, :] # splice to get the applicable entries
    df = df.drop(columns={'Burn %'}) # do not need the burn percentage
    df.dropna(axis=1, inplace=True, how='all') # can also drop the two NaN columns
    df.rename(columns={'AIR BASIN':'air_basin'}, inplace=True)
    return df

In [165]:
def melt_data(df, days_in_month):
    """Melt the dataframe so that columns include the air basin, day, and decision code, rather than having day as column labels"""
    days_in_month = days_in_month + 1 # stops at and does not include this value, so add 1
    return df.melt(id_vars=['air_basin'], value_vars=list(np.arange(1, days_in_month)),
                   var_name='day', value_name='burn_day_decision_code')

In [166]:
def assign_year_and_month(df, year, month):
    """Assign the year and month to all rows in the dataframe"""
    df['year'] = year
    df['month'] = month
    return df

In [167]:
def get_date(df):
    """Use pd datetime function to get the date"""
    df['date'] = pd.to_datetime(df[['year', 'day', 'month']]) # record date
    df = df.drop(columns={'day', 'year', 'month'}) # drop now unnecessary month, day, year cols
    return df

We can test the above functions on one sheet, as an example, before applying them to our entire dataset.

In [168]:
# Example of testing functionality for Jan, 2020

months = ['Jan', 'Feb', 'March', 'April', 'May', 'Jun', 'Jul',
          'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

jan_2020 = pd.read_excel('md2020.xlsx', sheet_name='Jan')

jan_2020_clean = clean_month(jan_2020)
jan_2020_melted = melt_data(jan_2020_clean, 31)

jan_2020_melted['month'] = 1
jan_2020_melted['year'] = 2020 # figure out the best way to compile this for each sheet

get_date(jan_2020_melted)

,air_basin,burn_day_decision_code,date
0,North Coast High,F,2020-01-01
1,North Coast Low,B,2020-01-01
2,Lake County,B,2020-01-01
3,San Francisco Bay North High,B,2020-01-01
4,San Francisco Bay North Low,B,2020-01-01
...,...,...,...
987,Lake Tahoe,B,2020-01-31
988,Great Basin Valleys North,B,2020-01-31
989,Great Basin Valleys South,B,2020-01-31
990,Bay Area Fall / Spring Tule Burn Allocation (A...,NaN,2020-01-31


Now, we can use the above functions to clean all of the datasheets and compile them into one dataframe.

In [169]:
def compile_data(datasheets):
    """Clean and compile individual datasheets into a singular dataframe"""
    
    # Initialize an empty list
    combined_data = []

    # Loop through all of the Excel workbooks/years
    for index, sheet in enumerate(datasheets):
    
        # Get the numeric year by summing index in list and 2020 -- for other years make note!
        num_year = index + 2020
    
        # Loop through all months to read individual sheets
        for index, month in enumerate(months):
        
            # Get the dataframe
            df = pd.read_excel(sheet, sheet_name=month)
        
            # Numeric month is the index shifted by 1
            num_month = index + 1
        
            # Clean the dataframe
            cleaned_df = clean_month(df)
        
            # Melt the dataframe (dependent on number of days per month)
            if num_month in set([1, 3, 5, 7, 8, 10, 12]):
                # Jan, March, May, July, August, Oct, and December all have 31 days
                melted_df = melt_data(cleaned_df, 31)
            elif num_month in set([4, 6, 9, 11]):
                # April, June, September, November all have 30 days
                melted_df = melt_data(cleaned_df, 30)
            elif num_month == 2:
                # February has 28
                melted_df = melt_data(cleaned_df, 28)
        
            # Assign the year and month to the df
            assigned_df = assign_year_and_month(melted_df, num_year, num_month)
        
            # Get the date of the dataframe
            dated_df = get_date(assigned_df)
        
            # Append to main list of sheets
            combined_data.append(dated_df)
    
    # concatenate all sheets together and return df
    return pd.concat(combined_data)

In [170]:
datasheets = ['md2020.xlsx', 'md2021.xlsx', 'md2022.xlsx',
              'md2023.xlsx', 'nc-md2024.xlsx']

all_years = compile_data(datasheets)

In [171]:
all_years

,air_basin,burn_day_decision_code,date
0,North Coast High,F,2020-01-01
1,North Coast Low,B,2020-01-01
2,Lake County,B,2020-01-01
3,San Francisco Bay North High,B,2020-01-01
4,San Francisco Bay North Low,B,2020-01-01
...,...,...,...
987,Lake Tahoe,B,2024-12-31
988,Great Basin Valleys North,B,2024-12-31
989,Great Basin Valleys South,B,2024-12-31
990,Bay Area Fall / Spring Tule Burn Allocation (A...,NaN,2024-12-31


Next, we need to associate the burn day decision codes with descriptions. Most will be straightforward, but we will need to specially handle the 'Sacramento Valley Low' and 'San Joaquin valley Authorized VOC / PM10 / NOx' entries.

In [172]:
# import dataframe that matches decision codes with descriptions
code_descriptors = pd.read_csv('code_descriptors.csv')
code_descriptors

,burn_day_decision_code,description
0,B,Permissive Burn Day Decision
1,M,Marginal Burn Day Decision
2,NB,No-Burn Day Decision
3,DB,Delayed Burn Day Decision
4,DN,Delayed No-Burn Day Decision
5,Ab,Decision Amended to Burn Day
6,An,Decision Amended to No-Burn Day
7,B^,"Burn Day, But Burn Restrictions Requested by C..."
8,M^,"Marginal as requested by CDF, USFS, or APCD"
9,S,Superior


In [173]:
# Join the two dataframes to associate codes with descriptors
all_with_descriptions = pd.merge(left=all_years, right=code_descriptors, on='burn_day_decision_code', how='left')
all_with_descriptions

,air_basin,burn_day_decision_code,date,description
0,North Coast High,F,2020-01-01,Fair
1,North Coast Low,B,2020-01-01,Permissive Burn Day Decision
2,Lake County,B,2020-01-01,Permissive Burn Day Decision
3,San Francisco Bay North High,B,2020-01-01,Permissive Burn Day Decision
4,San Francisco Bay North Low,B,2020-01-01,Permissive Burn Day Decision
...,...,...,...,...
58395,Lake Tahoe,B,2024-12-31,Permissive Burn Day Decision
58396,Great Basin Valleys North,B,2024-12-31,Permissive Burn Day Decision
58397,Great Basin Valleys South,B,2024-12-31,Permissive Burn Day Decision
58398,Bay Area Fall / Spring Tule Burn Allocation (A...,NaN,2024-12-31,NaN


There were two types of M, Marginal codes - if the entry is in one of the Select Air Basins, we should mark the description as 'Marginal' as opposed to 'Marginal Burn Day Decision'

In [174]:
# List of air basins to check
target_basins = ['North Coast High', 'Sacramento Valley High', 'Northeast Plateau', 'Mountain Counties North', 'Mountain Counties South']

# Update the 'description' where both conditions are true
all_with_descriptions.loc[
    (all_with_descriptions['air_basin'].isin(target_basins)) & (all_with_descriptions['burn_day_decision_code'] == 'M'),
    'description'
] = 'Marginal'

It isn't anywhere in the key, but we can specify Acres within the description of the Tule Burn Allocation based on the description

In [175]:
# Assign Bay Area Fall to have description of Acres
all_with_descriptions.loc[
    (all_with_descriptions['air_basin'] == 'Bay Area Fall / Spring Tule Burn Allocation (Acres)'), 'description'] = 'Acres'

Handle Sacramento Valley Low Codes independently, then add descriptions to main dataframe. We can do this by extracting the letters associated with the code and then adding the corresponding descriptions.

In [176]:
# Create a dataframe of only the Sacramento Valley Low values
sac_val_low = all_with_descriptions[all_with_descriptions['air_basin'] == 'Sacramento Valley Low']

# Extract letters from the decision codes, convert to uppercase to match with key
sac_val_low['code_str'] = sac_val_low['burn_day_decision_code'].str.extract('([a-zA-Z]+)')
sac_val_low['code_str'] = sac_val_low['code_str'].str.upper()

# Join with the descriptors table to assign descriptions
sac_val_low_with_desc = pd.merge(left=sac_val_low, right=code_descriptors, how='left', left_on='code_str', right_on='burn_day_decision_code')
sac_val_low_with_desc['description_sac'] = "The Number of Acres Allocated (in 1,000's)" + " - " + sac_val_low_with_desc['description_y']

# If there is no additional letter, just indicate the number of acres allocated
sac_val_low_with_desc['description_sac'].replace({None: "The Number of Acres Allocated (in 1,000's)"}, inplace=True)

# Get rid of redundant columns
sac_val_low_with_desc = sac_val_low_with_desc[['air_basin', 'date', 'description_sac']]
sac_val_low_with_desc

# Merge with main df
all_with_descriptions = pd.merge(left=all_with_descriptions, right=sac_val_low_with_desc, how='left', on=['air_basin', 'date'])
all_with_descriptions['description'] = all_with_descriptions['description'].fillna(all_with_descriptions['description_sac'])
all_with_descriptions.drop(columns=['description_sac'], inplace=True)

C:\Users\mhodgson\AppData\Local\Temp\ipykernel_23480\2894977793.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sac_val_low['code_str'] = sac_val_low['burn_day_decision_code'].str.extract('([a-zA-Z]+)')
C:\Users\mhodgson\AppData\Local\Temp\ipykernel_23480\2894977793.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sac_val_low['code_str'] = sac_val_low['code_str'].str.upper()
C:\Users\mhodgson\AppData\Local\Temp\ipykernel_23480\2894977793.py:13: FutureWarning: A value is trying to be set on a copy of 

Handle San Joaquin Valley entries, then add to main dataframe, similar to above:

In [177]:
# Create a separate df with the San Joaquin Valley entries
san_joa_valley = all_with_descriptions[all_with_descriptions['air_basin'] == 'San Joaquin Valley Authorized VOC / PM10 / NOx']

# Extract letters from the decision codes, convert to uppercase to match with key
san_joa_valley['code_str'] = san_joa_valley['burn_day_decision_code'].str.extract('([a-zA-Z]+)')
san_joa_valley['code_str'] = san_joa_valley['code_str'].str.upper()

# Create a working column for new description
san_joa_valley['desc'] = san_joa_valley['code_str'].replace({'P':'Number of Tons - Particulate Matter', 'N':'Number of Tons - NOx', None:'Number of Tons'})
san_joa_valley = san_joa_valley[['air_basin', 'date', 'desc']]

# Merge with main df
all_with_descriptions = pd.merge(left=all_with_descriptions, right=san_joa_valley, how='left', on=['air_basin', 'date'])
all_with_descriptions['description'] = all_with_descriptions['description'].fillna(all_with_descriptions['desc'])
all_with_descriptions.drop(columns=['desc'], inplace=True)

C:\Users\mhodgson\AppData\Local\Temp\ipykernel_23480\3455203699.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  san_joa_valley['code_str'] = san_joa_valley['burn_day_decision_code'].str.extract('([a-zA-Z]+)')
C:\Users\mhodgson\AppData\Local\Temp\ipykernel_23480\3455203699.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  san_joa_valley['code_str'] = san_joa_valley['code_str'].str.upper()
C:\Users\mhodgson\AppData\Local\Temp\ipykernel_23480\3455203699.py:9: SettingWithCopyWarning: 
A value is trying to

We have now compiled a database of all agricultural burning control notice code entries, as well as their corresponding descriptions:

In [178]:
all_with_descriptions.head()

,air_basin,burn_day_decision_code,date,description
0,North Coast High,F,2020-01-01,Fair
1,North Coast Low,B,2020-01-01,Permissive Burn Day Decision
2,Lake County,B,2020-01-01,Permissive Burn Day Decision
3,San Francisco Bay North High,B,2020-01-01,Permissive Burn Day Decision
4,San Francisco Bay North Low,B,2020-01-01,Permissive Burn Day Decision


In [180]:
all_with_descriptions.to_csv('burn_day_data.csv')